In [0]:
# 1. INSTALAÇÃO DE PACOTES EXTERNOS
# O Databricks já possui o PySpark nativo, mas garantimos as bibliotecas de gráficos e requisições web
%pip install plotly requests

# 2. IMPORTAÇÃO DE BIBLIOTECAS
# 2.1 PySpark SQL: Para higienização, filtros e manipulação de colunas
from pyspark.sql.functions import col, length, lpad, count, desc, when, to_date, datediff
from pyspark.sql.types import StringType, IntegerType

# 2.2 PySpark MLlib: Para a futura clusterização de prestadores (K-Means)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 2.3 Integração de APIs: Para consumir dados de CNPJ futuramente
import requests
import json

# 2.4 Visualização de Dados e Mapas Georreferenciados
import plotly.express as px
import plotly.graph_objects as go

In [0]:
# ==============================================================================
# BLOCO 2: INGESTÃO, HIGIENIZAÇÃO E GRAVAÇÃO DA CAMADA PRATA (TABELA DELTA)
# ==============================================================================

# 1. Definição do caminho do Volume criado no Unity Catalog
# O asterisco (*) garante a leitura concorrente de todos os 27 arquivos CSV
caminho_raw = "/Volumes/workspace/default/ans_prestadores_nao_hospitalares/*.csv"

# 2. Leitura Unificada do Conjunto de Dados
df_raw = spark.read.csv(caminho_raw, 
                        header=True, 
                        sep=";", 
                        inferSchema=True,
                        encoding="latin1")

# 3. Higienização e Aplicação do Filtro PJ (14 dígitos)
df_pj = df_raw \
    .withColumn("CNPJ_STR", col("CD_CNPJ_ESTB_SAUDE").cast("string")) \
    .withColumn("CNPJ_CLEAN", lpad(col("CNPJ_STR"), 14, "0")) \
    .filter(length(col("CNPJ_CLEAN")) == 14)

# 4. Gravação da Tabela Consolidada (Camada Prata) no Catálogo
nome_tabela_destino = "workspace.default.ans_rede_nao_hospitalar_pj"

df_pj.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela_destino)

# 5. Validação da Execução
total_bruto = df_raw.count()
total_pj = spark.table(nome_tabela_destino).count()

print(f"Ingestão concluída com sucesso!")
print(f"Total de registros na base bruta (PF+PJ): {total_bruto:,}")
print(f"Total de registros após higienização (Apenas PJ): {total_pj:,}")
print(f"Tabela Delta disponível em: {nome_tabela_destino}")

In [0]:
# ==============================================================================
# BLOCO 3: ANÁLISE GEOGRÁFICA (O ESPAÇO DO CUIDADO) - MAPA DE CALOR POR UF
# ==============================================================================

import json
import urllib.request
import plotly.express as px
from pyspark.sql.functions import countDistinct, col

# 1. Carregar a Tabela Delta (Camada Prata)
df_pj = spark.table("workspace.default.ans_rede_nao_hospitalar_pj")

# 2. Agregação Espacial: Contagem de Infraestruturas Únicas (CNPJ) por Estado
# Renomeado para "PRESTADORES" (inclui laboratórios, SADT, etc.)
df_uf = df_pj.groupBy("SG_UF").agg(
    countDistinct("CNPJ_CLEAN").alias("TOTAL_PRESTADORES_UNICOS")
).orderBy(col("TOTAL_PRESTADORES_UNICOS").desc())

# 3. Conversão para Pandas
pdf_uf = df_uf.toPandas()

# Exibe a tabela sintética para validação rápida
display(pdf_uf)

# 4. Importação da malha geográfica do Brasil (GeoJSON)
url_geojson = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
with urllib.request.urlopen(url_geojson) as response:
    brazil_geojson = json.load(response)

# 5. Plotagem do Mapa Coroplético (Identidade Visual Lactec: Tons de Azul)
fig = px.choropleth(
    pdf_uf,
    geojson=brazil_geojson,
    locations="SG_UF",
    featureidkey="properties.sigla", 
    color="TOTAL_PRESTADORES_UNICOS",
    color_continuous_scale="Blues", # Escala ajustada para a paleta do Lactec
    title="Distribuição Geográfica de Prestadores Não-Hospitalares (PJ)",
    labels={"TOTAL_PRESTADORES_UNICOS": "Prestadores Únicos"}
)

# Ajustes de layout
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    title_font_size=18,
    coloraxis_colorbar=dict(title="Qtd. Prestadores")
)

# Renderiza o gráfico
fig.show()

In [0]:
# 5. Plotagem de Gráfico de Barras Horizontais (Ideal para grandes disparidades)
fig2 = px.bar(
    pdf_terceirizacao, 
    x="TOTAL_VINCULOS", 
    y="TIPO_PRESTADOR_CLEAN",
    orientation='h', # Barras horizontais
    color="TIPO_PRESTADOR_CLEAN",
    color_discrete_map=color_map,
    text="TOTAL_VINCULOS", # Exibe o número exato na barra
    title="Grau de Terceirização da Malha Assistencial Não-Hospitalar (Vínculos PJ)",
    labels={
        "TOTAL_VINCULOS": "Total de Vínculos Contratuais",
        "TIPO_PRESTADOR_CLEAN": "Natureza do Vínculo"
    }
)

# Ajustes estéticos para legibilidade e padrão acadêmico
fig2.update_traces(
    texttemplate='%{text:,.0f}', # Formata os números com separador de milhar
    textposition='outside',      # Coloca o número fora da barra (para as barras curtas)
    cliponaxis=False             # Impede que números muito grandes sejam cortados na margem
)

fig2.update_layout(
    title_font_size=18,
    showlegend=False,
    margin=dict(t=50, b=40, l=120, r=60), # Margem extra à direita para caber os números
    xaxis=dict(title="", showticklabels=False), # Oculta o eixo X inferior para visualização mais limpa
    yaxis=dict(title="", categoryorder='total ascending') # Ordena da menor para a maior barra
)

fig2.show()

In [0]:
# ==============================================================================
# BLOCO 5: EXTRAÇÃO COM MULTI-THREADING (CONCORRÊNCIA CONTROLADA)
# ==============================================================================

import requests
import time
import pandas as pd
from pyspark.sql.functions import col, upper, translate
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Carregar a Tabela Delta e aplicar o Filtro da RMC (29 municípios)
df_pj = spark.table("workspace.default.ans_rede_nao_hospitalar_pj")

municipios_rmc = [
    "CURITIBA", "ADRIANOPOLIS", "AGUDOS DO SUL", "ALMIRANTE TAMANDARE", 
    "ARAUCARIA", "BALSA NOVA", "BOCAIUVA DO SUL", "CAMPINA GRANDE DO SUL", 
    "CAMPO DO TENENTE", "CAMPO LARGO", "CAMPO MAGRO", "CERRO AZUL", 
    "COLOMBO", "CONTENDA", "DOUTOR ULYSSES", "FAZENDA RIO GRANDE", 
    "ITAPERUCU", "LAPA", "MANDIRITUBA", "PIEN", "PINHAIS", "PIRAQUARA", 
    "QUATRO BARRAS", "QUITANDINHA", "RIO BRANCO DO SUL", "RIO NEGRO", 
    "SAO JOSE DOS PINHAIS", "TIJUCAS DO SUL", "TUNAS DO PARANA"
]

df_rmc = df_pj.filter(col("SG_UF") == "PR").withColumn(
    "NM_MUN_CLEAN", 
    translate(upper(col("NM_MUNICIPIO")), "ÁÀÂÃÉÈÊÍÏÓÔÕÖÚÇÑ", "AAAAEEEIIOOOOUCN")
).filter(col("NM_MUN_CLEAN").isin(municipios_rmc))

# 2. Extrair CNPJs únicos para lista Python
lista_cnpjs = df_rmc.select("CNPJ_CLEAN").distinct().toPandas()['CNPJ_CLEAN'].tolist()
total = len(lista_cnpjs)
print(f"Iniciando extração acelerada para {total} CNPJs...")

# 3. Função de consulta isolada com política de Retry (Retentativa)
def fetch_cnae(cnpj):
    url = f"https://brasilapi.com.br/api/cnpj/v1/{cnpj}"
    for tentativa in range(3): # Tenta até 3 vezes em caso de bloqueio (Rate Limit)
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                return {'CNPJ_CLEAN': cnpj, 'CNAE_PRINCIPAL': str(resp.json().get("cnae_fiscal", ""))}
            elif resp.status_code == 429: # Se a API pedir para desacelerar
                time.sleep(1.5) 
            else:
                return {'CNPJ_CLEAN': cnpj, 'CNAE_PRINCIPAL': "NAO_ENCONTRADO"}
        except:
            time.sleep(1)
    return {'CNPJ_CLEAN': cnpj, 'CNAE_PRINCIPAL': "ERRO"}

# 4. Multi-threading: Executando 15 requisições de forma simultânea
resultados_cnae = []
with ThreadPoolExecutor(max_workers=15) as executor:
    # Dispara todas as tarefas
    futures = {executor.submit(fetch_cnae, cnpj): cnpj for cnpj in lista_cnpjs}
    
    # Coleta os resultados conforme vão terminando
    for i, future in enumerate(as_completed(futures)):
        resultados_cnae.append(future.result())
        # Atualiza a tela a cada 500 registros
        if (i + 1) % 500 == 0:
            print(f"Processados {i + 1} de {total} CNPJs...")

# 5. Converter resultados, Filtrar Rede TEA e Salvar Tabela Delta
df_cnaes_extraidos = spark.createDataFrame(pd.DataFrame(resultados_cnae))
cnaes_tea = ["8650003", "8650005", "8650006", "8650004"]

df_cnpjs_tea = df_cnaes_extraidos.filter(col("CNAE_PRINCIPAL").isin(cnaes_tea))
df_rede_tea_rmc = df_rmc.join(df_cnpjs_tea, on="CNPJ_CLEAN", how="inner")

nome_tabela_final = "workspace.default.ans_malha_tea_rmc"
df_rede_tea_rmc.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome_tabela_final)

print(f"\nSucesso! Malha especializada estruturada e salva em {nome_tabela_final}")
display(df_rede_tea_rmc)

In [0]:
# ==============================================================================
# BLOCO 6: COMPOSIÇÃO DA REDE MULTIDISCIPLINAR TEA POR ESPECIALIDADE (CNAE)
# ==============================================================================

import plotly.express as px
import pandas as pd
from pyspark.sql.functions import col, countDistinct, when

# 1. Carregar a malha específica de TEA da RMC gerada e enriquecida via API
df_tea_rmc = spark.table("workspace.default.ans_malha_tea_rmc")

# 2. Traduzir os códigos CNAE para a nomenclatura comercial das especialidades
df_especialidades = df_tea_rmc.withColumn(
    "ESPECIALIDADE",
    when(col("CNAE_PRINCIPAL") == "8650003", "Psicologia")
    .when(col("CNAE_PRINCIPAL") == "8650006", "Fonoaudiologia")
    .when(col("CNAE_PRINCIPAL") == "8650005", "Terapia Ocupacional")
    .when(col("CNAE_PRINCIPAL") == "8650004", "Fisioterapia")
    .otherwise("Outras Atividades")
)

# 3. Agregação: Contar quantas clínicas únicas existem para cada especialidade na RMC
df_agrupado = df_especialidades.groupBy("ESPECIALIDADE").agg(
    countDistinct("CNPJ_CLEAN").alias("TOTAL_CLINICAS")
).orderBy(col("TOTAL_CLINICAS").desc())

# 4. Conversão para Pandas para renderização no Plotly
pdf_especialidades = df_agrupado.toPandas()

# 5. Plotagem do Gráfico de Barras Horizontais
fig4 = px.bar(
    pdf_especialidades,
    x="TOTAL_CLINICAS",
    y="ESPECIALIDADE",
    orientation='h', # Barras horizontais acomodam melhor os nomes das terapias
    text="TOTAL_CLINICAS",
    title="Composição por tipo de Terapia (Dados CNAE - CNPJ)",
    labels={
        "TOTAL_CLINICAS": "Total de Estabelecimentos (CNPJ)",
        "ESPECIALIDADE": "Especialidade Terapêutica"
    },
    color_discrete_sequence=["#004B79"] # Azul Lactec
)

# Ajustes estéticos
fig4.update_traces(
    texttemplate='%{text}',
    textposition='outside',
    cliponaxis=False
)
fig4.update_layout(
    title_font_size=16,
    margin=dict(t=50, b=40, l=150, r=40),
    xaxis=dict(title="", showticklabels=False), # Oculta eixo X para visualização mais limpa
    yaxis=dict(title="", categoryorder='total ascending') # Ordena as barras visualmente
)

fig4.show()

In [0]:
# ==============================================================================
# BLOCO 7 (CORRIGIDO): TABELA DE TERAPIAS COM OS 29 MUNICÍPIOS DA RMC
# ==============================================================================

from pyspark.sql.functions import col, countDistinct, when
from pyspark.sql import Row

# 1. Carregar a malha TEA da RMC
df_tea_rmc = spark.table("workspace.default.ans_malha_tea_rmc")

# 2. Traduzir CNAE para Especialidade
df_espec = df_tea_rmc.withColumn(
    "ESPECIALIDADE",
    when(col("CNAE_PRINCIPAL") == "8650003", "Psicologia")
    .when(col("CNAE_PRINCIPAL") == "8650006", "Fonoaudiologia")
    .when(col("CNAE_PRINCIPAL") == "8650005", "Terapia Ocupacional")
    .when(col("CNAE_PRINCIPAL") == "8650004", "Fisioterapia")
    .otherwise("Outras")
)

# 3. Pivotar a tabela com os dados existentes
df_pivot = df_espec.groupBy("NM_MUN_CLEAN").pivot("ESPECIALIDADE").agg(
    countDistinct("CNPJ_CLEAN")
).fillna(0)

# 4. Lista dos 29 Municípios Legais da RMC
municipios_rmc = [
    "CURITIBA", "ADRIANOPOLIS", "AGUDOS DO SUL", "ALMIRANTE TAMANDARE", 
    "ARAUCARIA", "BALSA NOVA", "BOCAIUVA DO SUL", "CAMPINA GRANDE DO SUL", 
    "CAMPO DO TENENTE", "CAMPO LARGO", "CAMPO MAGRO", "CERRO AZUL", 
    "COLOMBO", "CONTENDA", "DOUTOR ULYSSES", "FAZENDA RIO GRANDE", 
    "ITAPERUCU", "LAPA", "MANDIRITUBA", "PIEN", "PINHAIS", "PIRAQUARA", 
    "QUATRO BARRAS", "QUITANDINHA", "RIO BRANCO DO SUL", "RIO NEGRO", 
    "SAO JOSE DOS PINHAIS", "TIJUCAS DO SUL", "TUNAS DO PARANA"
]

# Cria um DataFrame apenas com a lista completa de cidades
df_todos_municipios = spark.createDataFrame([Row(NM_MUN_CLEAN=m) for m in municipios_rmc])

# 5. Cruzamento para forçar a exibição de todas as cidades (Left Join)
df_completo = df_todos_municipios.join(df_pivot, on="NM_MUN_CLEAN", how="left").fillna(0)

# 6. Adicionar coluna de Total para ordenação correta
colunas_terapias = [c for c in df_completo.columns if c != "NM_MUN_CLEAN"]
expressao_soma = sum(col(c) for c in colunas_terapias)
df_final = df_completo.withColumn("TOTAL_GERAL", expressao_soma)

# 7. Ordenar de forma decrescente pelo Total Geral
df_final = df_final.orderBy(col("TOTAL_GERAL").desc())

# 8. Exibir a tabela consolidada
display(df_final)